# AI-Based Ship Acquisition Delay Prediction System
### Complete Pipeline Demonstration: Data Generation, ML Training, Evaluation, Inference, and Explainability

This notebook walks through the full end-to-end machine learning pipeline for generating a synthetic ship procurement dataset, validating it, training regression and classification models, comparing performance, saving the best models, and running inference with rule-based recommendations.

## Environment Setup
Let's import requirements and append parent directory to python path to access pipeline modules.

In [1]:
import sys
import os
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append(str(Path(os.getcwd()).parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
print("Environment ready.")

Environment ready.


## Step 1: Generate Synthetic Dataset
We call `run_dataset_generation` which generates baseline features, applies correlations, computes delays, and registers risk categories.

In [2]:
from generate_dataset import run_dataset_generation

# Generate 1000 records for quick notebook run
df_raw = run_dataset_generation(size=1000, seed=42)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

ModuleNotFoundError: No module named 'generate_dataset'

## Step 2: Validate Dataset
We run the `DatasetValidator` to ensure all fields respect expected ranges, types, and logic constraints.

In [ ]:
from dataset_validator import DatasetValidator

validator = DatasetValidator()
df_validated, is_valid = validator.validate(df_raw, repair=True)
print(f"Is dataset valid after validation/repair step? {is_valid}")

## Step 3: Exploratory Data Analysis (EDA)
Let's look at numerical features stats and the distribution of targets.

In [ ]:
# Project stats
df_validated[['Project_Cost', 'Project_Size', 'Planned_Duration', 'Delay_Months', 'Delay_Percentage']].describe().T

In [ ]:
# Risk category count
df_validated['Risk_Category'].value_counts()

## Step 4: Visualize Data
Let's generate the key plots. We'll use the functions in `visualization.py` which saves them under `plots/`.

In [ ]:
from visualization import plot_correlation_heatmap, plot_delay_distribution, plot_risk_distribution

# Ensure plots dir exists in notebook context
plots_dir = Path('../plots')
plots_dir.mkdir(exist_ok=True)

plot_correlation_heatmap(df_validated, save_path=plots_dir / "correlation_heatmap.png")
plot_delay_distribution(df_validated, save_path=plots_dir / "delay_distribution.png")
plot_risk_distribution(df_validated, save_path=plots_dir / "risk_distribution.png")

Let's display the risk distribution countplot inline:

In [ ]:
img = plt.imread(str(plots_dir / "risk_distribution.png"))
plt.figure(figsize=(10, 8), dpi=100)
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 5: Preprocess Data and Engineer Features
We apply feature engineering to create interaction features, and preprocess to scale features and one-hot encode categorical features.

In [ ]:
from feature_engineering import engineer_features
from preprocess import Preprocessor
from sklearn.model_selection import train_test_split

# 1. Feature Engineering
df_engineered = engineer_features(df_validated)

# 2. Preprocess
preprocessor = Preprocessor()
X = preprocessor.fit_transform(df_engineered)
targets = preprocessor.extract_targets(df_engineered)

# Extract target vectors
y_pct = targets["Delay_Percentage"]
y_months = targets["Delay_Months"]
y_risk = targets["Risk_Category"]

# 3. Split into Train & Test (80:20)
X_train, X_test, y_train_pct, y_test_pct = train_test_split(X, y_pct, test_size=0.2, random_state=42)
_, _, y_train_months, y_test_months = train_test_split(X, y_months, test_size=0.2, random_state=42)
_, _, y_train_risk, y_test_risk = train_test_split(X, y_risk, test_size=0.2, random_state=42, stratify=y_risk)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## Step 6: Train and Evaluate Models
Train regressors (Decision Tree, Random Forest, Gradient Boosting, XGBoost) and classifiers (Decision Tree, Random Forest, Gradient Boosting, XGBoost).

In [ ]:
from train_models import train_regression_models, train_classification_models
from evaluate_models import compare_regression_models, compare_classification_models

# Train
reg_models = train_regression_models(X_train, y_train_pct, y_train_months, random_state=42)
cls_models = train_classification_models(X_train, y_train_risk, random_state=42)

Evaluate regression models predicting **Delay Percentage**:

In [ ]:
df_reg_pct_metrics, best_reg_pct_name, best_reg_pct_model = compare_regression_models(
    reg_models["Delay_Percentage"], X_test, y_test_pct
)

Evaluate regression models predicting **Delay Months**:

In [ ]:
df_reg_months_metrics, best_reg_months_name, best_reg_months_model = compare_regression_models(
    reg_models["Delay_Months"], X_test, y_test_months
)

Evaluate classification models predicting **Risk Category**:

In [ ]:
df_cls_metrics, best_cls_name, best_cls_model = compare_classification_models(
    cls_models, X_test, y_test_risk
)

## Step 7: Plot Model Diagnostics & Evaluation
Let's plot Confusion Matrix, ROC-AUC, Residuals, and Feature Importance of the best models.

In [ ]:
from visualization import plot_confusion_matrix, plot_roc_curve, plot_residual_plot, plot_feature_importance
from sklearn.metrics import confusion_matrix

# 1. Confusion Matrix for Best Classifier
y_pred_cls = best_cls_model.predict(X_test)
cm = confusion_matrix(y_test_risk, y_pred_cls)
classes_list = ["Low", "Medium", "High", "Critical"]
plot_confusion_matrix(cm, classes=classes_list, save_path=plots_dir / "confusion_matrix.png")

# 2. ROC Curve for Best Classifier
try:
    y_prob_cls = best_cls_model.predict_proba(X_test)
except AttributeError:
    y_prob_cls = np.eye(4)[y_pred_cls]
plot_roc_curve(y_test_risk, y_prob_cls, classes=classes_list, save_path=plots_dir / "roc_curve.png")

# 3. Residual Plot for Best Regressor
y_pred_pct = best_reg_pct_model.predict(X_test)
plot_residual_plot(y_test_pct, y_pred_pct, save_path=plots_dir / "residual_plot.png")

# 4. Feature Importance for Best Regressor
if hasattr(best_reg_pct_model, "feature_importances_"):
    plot_feature_importance(
        best_reg_pct_model.feature_importances_, 
        preprocessor.feature_names_out, 
        save_path=plots_dir / "feature_importance.png"
    )

Let's display the confusion matrix and feature importance inline:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=100)
axes[0].imshow(plt.imread(str(plots_dir / "confusion_matrix.png")))
axes[0].axis('off')
axes[0].set_title("Confusion Matrix")

if (plots_dir / "feature_importance.png").exists():
    axes[1].imshow(plt.imread(str(plots_dir / "feature_importance.png")))
    axes[1].axis('off')
    axes[1].set_title("Feature Importance")

plt.tight_layout()
plt.show()

## Step 8: Save Trained Models & Preprocessor
Save objects using `save_model.py`.

In [ ]:
from save_model import save_all_trained_models

save_all_trained_models(
    regression_models=reg_models,
    classification_models=cls_models,
    preprocessor=preprocessor,
    best_reg_pct_model=best_reg_pct_model,
    best_reg_months_model=best_reg_months_model,
    best_cls_model=best_cls_model
)

## Step 9: Predict & Explain on Custom Project
Input a hypothetical delayed project and inspect predictions, drivers, and recommendations.

In [ ]:
from predict import predict_project_delays
from explain_prediction import explain_prediction_details, print_explanation_report

# Custom input parameters containing severe bottlenecks
custom_project = {
    "Ship_Type": "Destroyer",
    "Project_Cost": 13500.0, # High cost Destroyer
    "Planned_Duration": 72,
    "Technical_Complexity": 8.5, # Very complex
    "Technology_Maturity": 5.0, # Immature tech
    "Foreign_Dependency": True, # Depends on imports
    "Approval_Delay": 135, # Severe administrative delay
    "Vendor_Delay": 120,   # Slow vendor responses
    "Requirement_Changes": 6 # Scope creep
}

# Run inference
prediction = predict_project_delays(custom_project)

# Run explanation & PMG steer rules
explanation = explain_prediction_details(custom_project, prediction)
print_explanation_report(explanation)